# 02 协同过滤（User-CF / Item-CF）
基于行为权重评分矩阵，实现基于用户和基于物品的协同过滤，并评估 Precision@K、Recall@K、NDCG@K

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# 读取评分矩阵（由 01_EDA 生成）
ratings = pd.read_csv('../data/csv/ratings.csv')
print(f'评分数: {len(ratings)}, 用户数: {ratings.user_id.nunique()}, 商品数: {ratings.product_id.nunique()}')

评分数: 18080, 用户数: 206, 商品数: 496


In [2]:
# ── 划分训练/测试集 ──────────────────────────────────────
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# 枚举索引映射
all_users    = sorted(ratings.user_id.unique())
all_products = sorted(ratings.product_id.unique())
u2i = {u: i for i, u in enumerate(all_users)}
p2i = {p: i for i, p in enumerate(all_products)}
i2p = {i: p for p, i in p2i.items()}
i2u = {i: u for u, i in u2i.items()}

# 稀疏评分矩阵（行=用户，列=商品）
rows = train.user_id.map(u2i)
cols = train.product_id.map(p2i)
R_train = csr_matrix((train.rating.values, (rows, cols)),
                     shape=(len(all_users), len(all_products)))
print('训练矩阵:', R_train.shape, '非零元素:', R_train.nnz)

训练矩阵: (206, 496) 非零元素: 14464


In [3]:
# ── 评估指标 ──────────────────────────────────────────────
def precision_recall_ndcg_at_k(recommended, relevant, k=10):
    rec_k = recommended[:k]
    hits  = len(set(rec_k) & set(relevant))
    precision = hits / k
    recall    = hits / len(relevant) if relevant else 0
    # NDCG
    dcg, idcg = 0.0, 0.0
    for i, item in enumerate(rec_k):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)
    for i in range(min(k, len(relevant))):
        idcg += 1 / np.log2(i + 2)
    ndcg = dcg / idcg if idcg > 0 else 0
    return precision, recall, ndcg

def evaluate_model(predict_fn, test_df, K=10, n_eval=50):
    """n_eval 减小到 50，评估更快"""
    test_users = test_df.user_id.unique()[:n_eval]
    ps, rs, ns = [], [], []
    for uid in test_users:
        relevant = test_df[test_df.user_id == uid].product_id.tolist()
        if not relevant:
            continue
        recs = predict_fn(uid, K * 2)
        p, r, n = precision_recall_ndcg_at_k(recs, relevant, K)
        ps.append(p); rs.append(r); ns.append(n)
    return np.mean(ps), np.mean(rs), np.mean(ns)

In [4]:
# ── Item-CF（相似度矩阵，向量化）──────────────────────────
# 注意：Item-CF 的物品相似矩阵 (n_items × n_items) 比 User-CF 小，更快
print('计算物品相似度矩阵...')
item_sim = cosine_similarity(R_train.T)   # (n_items, n_items)
print(f'item_sim shape: {item_sim.shape}')

# 向量化批量预测（替代逐用户循环）
R_dense = R_train.toarray()   # (n_users, n_items)

# Item-CF 预测矩阵：R_pred = R_train × item_sim  (n_users × n_items)
# 每一行 = 用户的加权物品相似分
R_item_pred = R_dense @ item_sim   # (n_users, n_items)

# 屏蔽已交互的商品（防止重复推荐）
R_item_pred[R_dense > 0] = -1

def item_cf_predict(uid, top_n=20):
    if uid not in u2i:
        return []
    ui = u2i[uid]
    scores = R_item_pred[ui]
    top_idx = np.argsort(scores)[::-1][:top_n]
    return [i2p[i] for i in top_idx]

p, r, n = evaluate_model(item_cf_predict, test, K=10)
print(f'Item-CF  Precision@10={p:.4f}  Recall@10={r:.4f}  NDCG@10={n:.4f}')

计算物品相似度矩阵...
item_sim shape: (496, 496)
Item-CF  Precision@10=0.0440  Recall@10=0.0244  NDCG@10=0.0460


In [5]:
# ── User-CF（向量化）──────────────────────────────────────
print('计算用户相似度矩阵...')
user_sim = cosine_similarity(R_train)   # (n_users, n_users)
print(f'user_sim shape: {user_sim.shape}')

# User-CF 预测矩阵：user_sim × R_train
R_user_pred = user_sim @ R_dense   # (n_users, n_items)
R_user_pred[R_dense > 0] = -1      # 屏蔽已交互

def user_cf_predict(uid, top_n=20):
    if uid not in u2i:
        return []
    ui = u2i[uid]
    scores = R_user_pred[ui]
    top_idx = np.argsort(scores)[::-1][:top_n]
    return [i2p[i] for i in top_idx]

p, r, n = evaluate_model(user_cf_predict, test, K=10)
print(f'User-CF  Precision@10={p:.4f}  Recall@10={r:.4f}  NDCG@10={n:.4f}')

计算用户相似度矩阵...
user_sim shape: (206, 206)
User-CF  Precision@10=0.0480  Recall@10=0.0258  NDCG@10=0.0504


In [6]:
# ── 保存 Item-CF 推荐结果（用于混合融合）──────────────────
# 向量化：直接从预测矩阵取 Top-50
cf_results = {}
top50_idx = np.argsort(R_item_pred, axis=1)[:, ::-1][:, :50]
for ui, uid in i2u.items():
    cf_results[uid] = [i2p[i] for i in top50_idx[ui]]

with open('../data/csv/cf_results.pkl', 'wb') as f:
    pickle.dump(cf_results, f)
print(f'Item-CF 推荐结果已保存（用户数: {len(cf_results)}）')

Item-CF 推荐结果已保存（用户数: 206）
